# Model Training and Evaluation

This notebook handles training and evaluating machine learning models for sentiment analysis.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load processed data
data = pd.read_csv('../data/processed/reviews_clean.csv')
print(f"Loaded {len(data)} processed reviews")
print(data.head())
print("\nDataset info:")
print(data.info())

In [ ]:
# Prepare features and target
X = data['review_text_clean']
y = data['sentiment_label']

print(f"Features shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining set distribution:")
print(y_train.value_counts())
print(f"\nTest set distribution:")
print(y_test.value_counts())

In [ ]:
# Text vectorization using TF-IDF
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    lowercase=True
)

# Fit and transform the training data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF features shape: {X_train_tfidf.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

In [ ]:
# Define models to train
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='linear', random_state=42)
}

# Train and evaluate models
model_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train_tfidf, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_tfidf)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    # Store results
    model_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'predictions': y_pred
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Classification Report:")
    print(classification_report(y_test, y_pred))

In [ ]:
# Compare model performances
accuracies = {name: results['accuracy'] for name, results in model_results.items()}

plt.figure(figsize=(10, 6))
models_names = list(accuracies.keys())
accuracy_scores = list(accuracies.values())

bars = plt.bar(models_names, accuracy_scores, color=['skyblue', 'lightgreen', 'lightcoral', 'gold'])
plt.title('Model Comparison - Accuracy Scores')
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.ylim(0, 1)

# Add accuracy labels on bars
for bar, acc in zip(bars, accuracy_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{acc:.3f}', ha='center', va='bottom')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Print best model
best_model_name = max(accuracies, key=accuracies.get)
print(f"\nBest performing model: {best_model_name} with accuracy: {accuracies[best_model_name]:.4f}")

In [ ]:
# Detailed evaluation of the best model
best_model = model_results[best_model_name]['model']
best_predictions = model_results[best_model_name]['predictions']

print(f"Detailed evaluation for {best_model_name}:")
print("\nClassification Report:")
print(classification_report(y_test, best_predictions))

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, best_predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=best_model.classes_, yticklabels=best_model.classes_)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# Hyperparameter tuning for the best model (example with Logistic Regression)
if best_model_name == 'Logistic Regression':
    print("Performing hyperparameter tuning for Logistic Regression...")
    
    param_grid = {
        'C': [0.1, 1, 10],
        'solver': ['liblinear', 'lbfgs']
    }
    
    grid_search = GridSearchCV(
        LogisticRegression(random_state=42, max_iter=1000),
        param_grid,
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )
    
    grid_search.fit(X_train_tfidf, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    # Update best model with tuned parameters
    best_model = grid_search.best_estimator_
    tuned_predictions = best_model.predict(X_test_tfidf)
    tuned_accuracy = accuracy_score(y_test, tuned_predictions)
    
    print(f"Tuned model accuracy: {tuned_accuracy:.4f}")

In [ ]:
# Feature importance analysis (for applicable models)
if hasattr(best_model, 'feature_names_in_') or hasattr(best_model, 'coef_'):
    print("\nAnalyzing feature importance...")
    
    if hasattr(best_model, 'coef_'):
        # For linear models like Logistic Regression
        feature_names = vectorizer.get_feature_names_out()
        
        # Get coefficients for each class
        if len(best_model.classes_) > 2:
            # Multi-class case
            for i, class_name in enumerate(best_model.classes_):
                coef = best_model.coef_[i]
                top_indices = np.argsort(np.abs(coef))[-10:]
                top_features = [(feature_names[idx], coef[idx]) for idx in top_indices]
                
                print(f"\nTop features for {class_name}:")
                for feature, importance in reversed(top_features):
                    print(f"{feature}: {importance:.4f}")
        else:
            # Binary case
            coef = best_model.coef_[0]
            top_indices = np.argsort(np.abs(coef))[-15:]
            top_features = [(feature_names[idx], coef[idx]) for idx in top_indices]
            
            print("\nTop 15 most important features:")
            for feature, importance in reversed(top_features):
                print(f"{feature}: {importance:.4f}")

In [ ]:
# Save the trained model and vectorizer
print("Saving trained model and vectorizer...")

# Save the best model
joblib.dump(best_model, '../model/sentiment_model.pkl')
print(f"Saved best model ({best_model_name}) to ../model/sentiment_model.pkl")

# Save the vectorizer
joblib.dump(vectorizer, '../model/vectorizer.pkl')
print("Saved vectorizer to ../model/vectorizer.pkl")

# Save model metadata
model_info = {
    'model_name': best_model_name,
    'accuracy': accuracies[best_model_name],
    'feature_count': X_train_tfidf.shape[1],
    'training_size': len(X_train),
    'test_size': len(X_test),
    'classes': list(best_model.classes_)
}

import json
with open('../model/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("Saved model metadata to ../model/model_info.json")
print("\nModel training completed successfully!")

In [ ]:
# Test the saved model
print("Testing saved model...")

# Load the saved model and vectorizer
loaded_model = joblib.load('../model/sentiment_model.pkl')
loaded_vectorizer = joblib.load('../model/vectorizer.pkl')

# Test with sample texts
sample_texts = [
    "This product is absolutely amazing! I love it!",
    "Terrible quality, waste of money. Very disappointed.",
    "It's okay, nothing special but works fine."
]

# Preprocess and predict
sample_tfidf = loaded_vectorizer.transform(sample_texts)
predictions = loaded_model.predict(sample_tfidf)
probabilities = loaded_model.predict_proba(sample_tfidf)

print("\nSample predictions:")
for i, (text, pred, prob) in enumerate(zip(sample_texts, predictions, probabilities)):
    max_prob = np.max(prob)
    print(f"Text: {text}")
    print(f"Predicted sentiment: {pred} (confidence: {max_prob:.3f})")
    print("-" * 50)